In [0]:
display(spark.sql("""
    SELECT
        regexp_extract(source_file, '[^/]+$', 0) AS archivo,
        COUNT(*) AS filas,
        SUM(
            CASE
                WHEN _rescued_data IS NOT NULL THEN 1
                ELSE 0
            END
        ) AS filas_para_revisar
    FROM workspace.default.bronze_pos_raw
    GROUP BY source_file
    ORDER BY archivo
"""))

In [0]:
total = spark.table(
    "workspace.default.bronze_pos_raw"
).count()

print(f"Filas actuales en Bronze: {total:,}")

In [0]:
display(spark.sql("""
    SELECT
        COUNT(*) AS filas_silver,
        SUM(CASE WHEN fecha IS NULL THEN 1 ELSE 0 END)
            AS fechas_no_interpretables,
        SUM(CASE WHEN hora_inicio IS NULL THEN 1 ELSE 0 END)
            AS horas_inicio_no_interpretables,
        SUM(CASE WHEN error_conversion_numerica THEN 1 ELSE 0 END)
            AS filas_con_error_numerico
    FROM workspace.default.silver_pos_lines
"""))

In [0]:
display(spark.sql("""
    SELECT
        COUNT(*) AS pedidos,
        SUM(lineas) AS lineas_agrupadas,
        SUM(
            CASE WHEN total_distintos <> 1 THEN 1 ELSE 0 END
        ) AS pedidos_sin_total_unico,
        SUM(
            CASE WHEN lineas_incompletas > 0 THEN 1 ELSE 0 END
        ) AS pedidos_con_lineas_incompletas,
        SUM(
            CASE
                WHEN impuesto_distintos > 1
                  OR descuento_distintos > 1
                THEN 1 ELSE 0
            END
        ) AS pedidos_con_campos_inconsistentes
    FROM workspace.default.silver_orders_base
"""))

In [0]:
display(spark.sql("""
    SELECT
        regla_total,
        COUNT(*) AS pedidos,
        COUNT(importe_comparable) AS pedidos_con_importe,
        COUNT(importe_comparable_estricto) AS pedidos_estrictos
    FROM workspace.default.silver_orders
    GROUP BY regla_total
    ORDER BY pedidos DESC
"""))

In [0]:
%sql
SELECT
    regexp_extract(source_file, '[^/]+$', 0) AS archivo,
    COUNT(*) AS filas,
    SUM(
        CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END
    ) AS filas_para_revisar
FROM workspace.default.bronze_pos_raw
GROUP BY source_file
ORDER BY archivo;

In [0]:
%sql
SELECT
    regexp_extract(source_file, '[^/]+$', 0) AS archivo,
    COUNT(*) AS filas,

    SUM(CASE
        WHEN fecha IS NULL THEN 1 ELSE 0
    END) AS fechas_invalidas,

    SUM(CASE
        WHEN hora_inicio IS NULL THEN 1 ELSE 0
    END) AS horas_invalidas,

    SUM(CASE
        WHEN error_conversion_numerica THEN 1 ELSE 0
    END) AS errores_numericos,

    SUM(CASE
        WHEN store_number IS NULL
          OR fecha IS NULL
          OR check_number IS NULL
          OR begin_time IS NULL
        THEN 1 ELSE 0
    END) AS claves_incompletas

FROM workspace.default.silver_pos_lines
GROUP BY source_file
ORDER BY archivo;

In [0]:
%sql
SELECT
    YEAR(fecha) AS anio,
    regla_total AS regla,
    COUNT(*) AS pedidos,
    COUNT(importe_comparable) AS con_importe
FROM workspace.default.silver_orders
GROUP BY YEAR(fecha), regla_total
ORDER BY anio, regla;

In [0]:
%sql
SELECT
    'Silver' AS capa,
    COUNT(*) AS pedidos,
    COUNT(importe_comparable) AS con_importe,
    ROUND(SUM(importe_comparable), 2) AS importe
FROM workspace.default.silver_orders
WHERE total_distintos = 1

UNION ALL

SELECT
    'Gold' AS capa,
    SUM(pedidos) AS pedidos,
    SUM(pedidos_con_importe) AS con_importe,
    ROUND(SUM(importe_estimado), 2) AS importe
FROM workspace.default.gold_store_daily;

In [0]:
%sql
SELECT
    fecha,
    pedidos,
    pedidos_sin_importe,
    cobertura_pct
FROM workspace.default.gold_order_quality_daily
ORDER BY fecha;

In [0]:
%sql
SELECT
    regla_total AS motivo,
    COUNT(*) AS pedidos
FROM workspace.default.silver_orders_review
GROUP BY regla_total
ORDER BY pedidos DESC;

In [0]:
%sql
WITH ultima_revision AS (
    SELECT
        ejecucion_id,
        MAX(revisado_at) AS fecha_revision
    FROM workspace.default.audit_source_files
    GROUP BY ejecucion_id
    ORDER BY fecha_revision DESC
    LIMIT 1
),
inventario AS (
    SELECT *
    FROM workspace.default.audit_source_files
    WHERE ejecucion_id = (
        SELECT ejecucion_id FROM ultima_revision
    )
),
cargados AS (
    SELECT DISTINCT
        regexp_replace(source_file, '^dbfs:', '') AS ruta
    FROM workspace.default.bronze_pos_raw
)
SELECT
    i.estado,
    i.columnas,
    COUNT(*) AS archivos,
    SUM(
        CASE
            WHEN i.estado = 'Con datos' AND c.ruta IS NULL
            THEN 1 ELSE 0
        END
    ) AS pendientes
FROM inventario i
LEFT JOIN cargados c
    ON regexp_replace(i.ruta, '^dbfs:', '') = c.ruta
GROUP BY i.estado, i.columnas
ORDER BY i.estado, i.columnas;